## 6. Solution method

### 6.1 Chosen method and justification

We solve the problem with an adaptive large neighbourhood search (ALNS) [15]. A Clarke-Wright savings solution [14] gives the starting point, a granular local search [17] improves the solutions the loop produces, and a simulated annealing rule [20] decides which candidates are kept.

A heuristic is needed because the problem is NP-hard (Section 4) and the ILP of Section 2 only closes small instances, while the project targets several thousand customers. ALNS fits this problem for four reasons.

* Destroy and repair change many customers in one step, so the search can leave regions that moves of a single customer cannot leave.
* The repair step is an insertion procedure. Under time windows an insertion is tested with the evaluation rule of Section 2.6, and the same test serves construction, repair and local search. The search therefore stays in the feasible space and needs no penalty terms.
* Granular neighbour lists restrict insertions and moves to nearby customers, which keeps the cost of an iteration bounded when $n$ is large.
* Operators are chosen according to their recent results, so we do not have to decide in advance which rule suits an instance.

Pan, Zhang and Lim [18] solve a closely related problem, the time-dependent VRPTW, with a hybrid of ALNS and tabu search. They report that a multi-start tabu variant performs worse than the hybrid, and that a variant restricted to feasible solutions loses little. Their objective and run times differ from ours, so we read this as support for the design and not as a prediction for our instances.

Other methods were considered. Ant colony optimisation needs an $n\times n$ pheromone matrix. Particle swarm optimisation works in a continuous space, while a route is an ordered list. A genetic algorithm stores a population, and crossover easily breaks time windows. Simulated annealing alone has no operator that moves many customers at once, so we use it only as the acceptance rule. Tabu search and hill climbing serve as baselines in the experimental study.

### 6.2 Solutions and feasibility

A solution is a set of routes $S=\{R_1,\dots,R_k\}$ that partitions the customer set $N$. A route $R=(i_1,\dots,i_l)$ stands for the path $0\to i_1\to\dots\to i_l\to n+1$. Departure times are not stored as decisions. They are recomputed from the route with the evaluation rule below.

*Evaluation rule.* For an arc $(i,j)$ of a route, with ready time $r_i=s_i+d_i$ and $s_0=0$,
$$\text{arr}_{ij}(r_i)=\min_{p:\ \max(r_i,\theta_{p-1})\le\theta_p}\Big\{\max(r_i,\theta_{p-1})+t_{ij}^p\Big\},\qquad s_j=\max\{a_j,\ \text{arr}_{ij}(r_i)\}.$$

*Cost.* $f(S)=\sum_{R\in S}\sum_{h=0}^{l}c_{i_hi_{h+1}}$, which equals the objective $Z$ of Section 2.

*Feasibility.* $S$ is feasible if $k\le m$ and every route satisfies the load limit $\sum_h q_{i_h}\le Q$, every start time $s_{i_h}\le b_{i_h}$, and the return time $s_{n+1}\le H$. By Section 2.7, a feasible $S$ corresponds to a solution of the ILP.

*Partial solution.* A set of feasible routes together with a pool $U$ of customers not yet served. It is the object that destroy operators produce and repair operators complete.

### 6.3 Initial solution

*Clarke-Wright savings* [14]. Each customer starts in its own route. For two customers $i$ and $j$ the saving is $\sigma_{ij}=c_{i,n+1}+c_{0j}-c_{ij}$. Savings are computed only between a customer and the members of its neighbour list (6.4) and sorted in decreasing order. Two routes are merged when $i$ ends one route, $j$ starts another, and the merged route is feasible. Routes are never reversed, because time windows fix their direction. The savings use the distances $c_{ij}$ as a static proxy for the time-dependent travel times.

*Fleet bound.* Since $m$ is a hard bound, a savings solution with more than $m$ routes is not admissible. In that case a route elimination step empties the smallest route into the pool $U$ and reinserts its customers with the repair operators of 6.5. If this fails, the run reports that the bound could not be met. This does not prove that the instance is infeasible.

### 6.4 Neighbourhood structure

*Granular neighbour lists.* For each customer $i$, the list $\mathcal{N}_\kappa(i)$ holds the $\kappa$ closest customers that are compatible with $i$ in time. The lists are built once. Every insertion and every local search move connects a customer only with a member of its list, as in the granular tabu search of Toth and Vigo [17]. The number of candidate moves per customer is then $O(\kappa)$ instead of $O(n)$.

*Large neighbourhood.* A neighbour $S'$ of $S$ is obtained in three steps: a destroy operator removes $\nu$ customers into the pool $U$, a repair operator reinserts all of them, and a local search improves the changed routes. The number $\nu$ is bounded so that the cost of one iteration stays limited on large instances.

*Local search neighbourhoods.* Relocate, swap, Or-opt and 2-opt*, all restricted to $\mathcal{N}_\kappa$ and all subject to the feasibility test of 6.2.

### 6.5 Operators

*Destroy operators.* Each maps a solution $S$ and a number $\nu$ to a partial solution $(S',U)$ with $|U|=\nu$.

* Random removal picks customers uniformly. It adds diversity.
* Worst removal favours customers whose removal saves the most distance, $\Delta_i=c_{pi}+c_{iv}-c_{pv}$ for predecessor $p$ and successor $v$. A random element keeps it from choosing the same customers every time.
* Shaw removal [16] removes a customer and then customers related to it, where relatedness combines distance, difference of service start times and difference of demands. The removed group can then be exchanged between routes.

*Repair operators.* Each maps a partial solution $(S',U)$ to a solution. Customer $u$ is inserted between $i$ and $j$ at cost $\Delta_u=c_{iu}+c_{uj}-c_{ij}$, only in feasible positions and only in routes that contain a member of $\mathcal{N}_\kappa(u)$, or in a new route if $k<m$.

* Greedy insertion repeatedly inserts the pool customer with the cheapest feasible position.
* Regret-2 insertion repeatedly inserts the customer with the largest difference between its second-best and best route. A customer with a single feasible route goes first. This protects customers with few options from losing their only slot.

If a customer has no feasible position, the candidate is rejected.

*Local search moves.* Relocate moves one customer to another position. Swap exchanges two customers. Or-opt moves a chain of two or three consecutive customers. 2-opt* exchanges the tails of two routes [19], and each tail keeps its direction. Plain 2-opt is not used, because reversing a segment changes the arrival time at every customer in it and all their windows would have to be rechecked.

### 6.6 Guidance mechanisms

*Adaptive operator selection* [15]. Each destroy and each repair operator has a weight. Operators are drawn by roulette wheel in proportion to their weights. During a segment of iterations, an operator earns a score when it produces a new best solution, an improvement, or an accepted worse solution. At the end of the segment the weights are updated from the average score per use, so the search shifts toward the operators that work on the current instance.

*Acceptance by simulated annealing* [20]. A candidate $S'$ replaces the current solution $S$ if $f(S')<f(S)$, and otherwise with probability $\exp\{-(f(S')-f(S))/T\}$. The temperature $T$ decreases during the run. Early in the run the search accepts small deteriorations and explores, and late in the run it accepts almost only improvements.

*Restart.* After a long period without improvement, the current solution is reset to the best one found.

Local search, greedy and regret insertion, and the low final temperature intensify the search. Random and Shaw removal, the high initial temperature and the adaptive weights diversify it.

### 6.7 Algorithm

The method has three stages. Initialisation builds the neighbour lists and the starting solution. The main loop applies one destroy, repair, local search and acceptance step per iteration. The loop stops at a time limit or after a number of iterations without improvement, and a final local search is applied to the best solution.

```
Algorithm 1: ALNS for the CVRPTW with peak traffic
Input: instance, parameters, random seed
 1  build neighbour lists N_kappa(i)
 2  S <- ClarkeWright()
 3  if k(S) > m: S <- RouteElimination(S)
 4  S_best <- S
 5  while stopping criterion not met:
 6      draw destroy operator d and repair operator r by roulette wheel
 7      (S', U) <- d(S) ; S' <- r(S', U)
 8      if U is not empty: reject, continue
 9      S' <- LocalSearch(S')
10      if f(S') < f(S_best): S_best <- S'
11      S <- S' if accepted by the annealing rule
12      update temperature, scores and weights
13      restart from S_best if no improvement for a long period
14  S_best <- LocalSearch(S_best)
15  return S_best
```

Every returned solution is checked with the evaluation rule of 6.2, so it satisfies the ILP constraints of Section 2. The run is repeated with several random seeds per instance.

### 6.8 Link with the exact model and limitations

The ILP and the heuristic describe the same feasible solutions through the evaluation rule (Section 2.7). On small instances the ILP gives the optimal cost, so the optimality gap of the heuristic can be computed exactly. With one traffic slot, open windows and zero service times the method solves the plain CVRP, which allows comparison with best-known values on vrplib instances.

The method gives no optimality guarantee, and its quality is measured by gaps. It searches only feasible solutions, so regions reachable only through infeasible intermediate solutions are not visited. The neighbour lists mean that a good move between two distant customers is never tried. Travel times are step functions fixed by the departure slot, which can differ from models with speeds that change during a trip [10].

### References cited in Section 6

Numbers follow Section 2.

10. Ichoua, S., Gendreau, M. and Potvin, J.-Y. (2003). Vehicle dispatching with time-dependent travel times. *European Journal of Operational Research*, 144(2), 379-396.
14. Clarke, G. and Wright, J. W. (1964). Scheduling of vehicles from a central depot to a number of delivery points. *Operations Research*, 12(4), 568-581.
15. Ropke, S. and Pisinger, D. (2006). An adaptive large neighborhood search heuristic for the pickup and delivery problem with time windows. *Transportation Science*, 40(4), 455-472.
16. Shaw, P. (1998). Using constraint programming and local search methods to solve vehicle routing problems. *CP98*, LNCS 1520, Springer, 417-431.
17. Toth, P. and Vigo, D. (2003). The granular tabu search and its application to the vehicle-routing problem. *INFORMS Journal on Computing*, 15(4), 333-346.
18. Pan, B., Zhang, Z. and Lim, A. (2020). A hybrid algorithm for time-dependent vehicle routing problem with time windows. Preprint submitted to *Computers & Operations Research*.
19. Potvin, J.-Y. and Rousseau, J.-M. (1995). An exchange heuristic for routeing problems with time windows. *Journal of the Operational Research Society*, 46(12), 1433-1446.
20. Kirkpatrick, S., Gelatt, C. D. and Vecchi, M. P. (1983). Optimization by simulated annealing. *Science*, 220(4598), 671-680.